# Level 2: Query Intelligence

This notebook implements the advanced routing pipeline required for Level 2.
Pipeline steps: Original query -> Query rewriting -> Query classification -> Filter extraction -> Filtered retrieval -> Answer generation

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Setup & API Keys
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Initialize models
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GEMINI_API_KEY, temperature=0.1)

# Load dataset and chunk it
with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

docs = []
for item in dataset:
    doc = Document(page_content=item['content'], metadata={"title": item['title'], "url": item['url']})
    docs.append(doc)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
all_chunks = text_splitter.split_documents(docs)
available_titles = list(set([doc.metadata['title'] for doc in all_chunks]))

C:\Users\Amira Roshdy\AppData\Local\Temp\ipykernel_11728\615408442.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Query Rewriting, Classification & Filter Extraction

In [2]:
# Prompt for Query Analysis
analyzer_template = """You are an expert AI search analyzer. Analyze the user's query about Azure.

Available Azure Product Titles:
{titles}

Your tasks:
1. "rewritten_query": Rewrite the query to be a highly descriptive, self-contained search query. Fix typos.
2. "intent": Classify the intent strictly as one of: [factual_lookup, comparison, recommendation, out_of_scope]
3. "title_filter": If the user is specifically asking about a product in the "Available Azure Product Titles" list, output the exact title string. If they are asking a general question, output null.

Return ONLY a valid JSON object. Do not include markdown formatting.
{{
  "rewritten_query": "...",
  "intent": "...",
  "title_filter": "..."
}}

Original Query: {query}
"""

analyzer_prompt = PromptTemplate.from_template(analyzer_template)

def analyze_query(query):
    titles_str = "\n".join([f"- {t}" for t in available_titles])
    chain = analyzer_prompt | llm | StrOutputParser()
    result = chain.invoke({"titles": titles_str, "query": query})
    result = result.replace("```json", "").replace("```", "").strip()
    return json.loads(result)

# Test 3 different query categories for the rubric!
test_queries = [
    "What is the difrence between azure functions and azure logic apps?", # Comparison
    "how to create a database in azure postgresql", # Factual lookup
    "how to bake a chocolate cake" # Out of scope
]

for q in test_queries:
    print(f"\n--- Raw Query: {q} ---")
    analysis = analyze_query(q)
    print(json.dumps(analysis, indent=2))


--- Raw Query: What is the difrence between azure functions and azure logic apps? ---
{
  "rewritten_query": "What is the difference between Azure Functions and Azure Logic Apps?",
  "intent": "comparison",
  "title_filter": null
}

--- Raw Query: how to create a database in azure postgresql ---
{
  "rewritten_query": "How to create a new database instance within Azure Database for PostgreSQL.",
  "intent": "factual_lookup",
  "title_filter": "Azure Database for PostgreSQL documentation"
}

--- Raw Query: how to bake a chocolate cake ---
{
  "rewritten_query": "How to bake a chocolate cake recipe and instructions.",
  "intent": "out_of_scope",
  "title_filter": null
}


## Filtered Retrieval & Answer Generation

In [3]:
def process_end_to_end(raw_query):
    print(f"\n==========================\nProcessing: {raw_query}")
    analysis = analyze_query(raw_query)
    
    if analysis['intent'] == 'out_of_scope':
        return "I am an Azure assistant. I cannot answer questions outside of Azure documentation."
        
    query_to_search = analysis['rewritten_query']
    title_filter = analysis['title_filter']
    
    # 4. Filtered Retrieval
    if title_filter:
        filtered_chunks = [c for c in all_chunks if c.metadata['title'] == title_filter]
    else:
        filtered_chunks = all_chunks
        
    if not filtered_chunks:
        return "No relevant documents found for filtering."
        
    temp_vector_store = FAISS.from_documents(filtered_chunks, embeddings)
    retrieved_docs = temp_vector_store.similarity_search(query_to_search, k=3)
    
    # 5. Answer Generation
    answer_template = """You are an expert AI assistant.
Use the following extremely relevant, filtered context to answer the question.
If you don't know the answer, say you don't know. Do NOT hallucinate.

Context:
{context}

Question: {question}

Answer:"""

    answer_prompt = PromptTemplate.from_template(answer_template)
    context_str = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    final_chain = answer_prompt | llm | StrOutputParser()
    return final_chain.invoke({
        "context": context_str, 
        "question": query_to_search
    })

# Test the factual query end-to-end
final_answer = process_end_to_end("how to create a database in azure postgresql")
print(f"\nFINAL ANSWER:\n{final_answer}")


Processing: how to create a database in azure postgresql

FINAL ANSWER:
To create a database instance in Azure Database for PostgreSQL, you can follow the "Quickstart Create an Azure Database for PostgreSQL" guide.
